In [1]:
from langgraph.graph import  START, END,StateGraph
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

e:\python\Langgraph\myenv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", api_key=os.getenv('API_KEY'))

In [3]:
class LLMState(TypedDict):
    title: str
    output: str
    content: str

In [4]:
def create_outline(state: LLMState) -> LLMState:
    title = state['title']

    prompt = f'Create an outline for a blog post with the title: {title}'

    outline = model.invoke(prompt).content

    state['output'] = outline

    return state



In [5]:
def create_blog(state: LLMState) -> LLMState:
    title = state['title']
    outline = state['output']

    prompt = f'Write a blog post with the title: {title} and the following outline: {outline}'

    blog = model.invoke(prompt).content

    state['content'] = blog

    return state

In [7]:
graph = StateGraph(LLMState)

graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)

graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", END)

workflow = graph.compile()


In [8]:
intial_state = LLMState(title="The Future of AI in India", output="", content="")
final_state = workflow.invoke(intial_state)
print(final_state['content'])

[{'type': 'text', 'text': '# The Future of AI in India: From the World’s Back Office to the World’s AI Laboratory\n\nFor decades, India was known as the "world’s back office"—the destination of choice for outsourced IT services and business processing. But as we move into 2024, a seismic shift is occurring. India is no longer just maintaining the world’s software; it is building the world’s intelligence.\n\nWith a massive digital footprint—powered by the world\'s cheapest data, the revolutionary UPI payment system, and the Aadhaar digital identity—India is uniquely positioned to lead the global AI revolution. By blending a "frugal innovation" mindset with a philosophy of **"AI for All,"** India is transforming from a service provider into a global AI powerhouse.\n\n## 1. The Government’s Vision: "AI for All"\n\nThe Indian government isn\'t just watching the AI wave; it’s steering it. The **NITI Aayog’s National Strategy for AI** and the recently approved **IndiaAI Mission** (with a bud

In [13]:
print(final_state['output'])

[{'type': 'text', 'text': 'This outline for a blog post titled **"The Future of AI in India"** is designed to be comprehensive, engaging, and structured for SEO.\n\n---\n\n# Blog Post Outline: The Future of AI in India\n\n## 1. Introduction\n*   **The Hook:** Mention India’s transition from being the "world’s back office" to becoming the "world’s AI laboratory."\n*   **Current Context:** Brief overview of the current AI wave (Generative AI) and India’s massive digital footprint (UPI, Aadhaar, cheap data).\n*   **Thesis Statement:** India is uniquely positioned to lead the global AI revolution through its "AI for All" philosophy, a booming startup ecosystem, and government-backed initiatives.\n\n## 2. The Government’s Vision: "AI for All"\n*   **National Strategy for AI:** Discuss NITI Aayog’s roadmap and the "IndiaAI" Mission.\n*   **Public Infrastructure:** How the **India Stack** (digital identity and payment systems) is evolving to include an AI layer.\n*   **Sovereign AI:** Mention